# ♟️ Benchmark 3 — Tournoi et classement Elo

Des vraies parties :
1. **tournoi toutes-rondes** entre les modèles (même recherche profondeur 2 pour les réseaux, + PWN alpha-bêta) ;
2. **matchs contre Stockfish 17 bridé** (`UCI_LimitStrength` à 1320 / 1600 / 1900 Elo, et `Skill Level 0` comme dans `stockfish-VS-PWN-prof2/`).

Chaque paire joue chaque **ouverture** de la liste deux fois (une fois avec chaque couleur) : les moteurs sont déterministes,
donc les ouvertures imposées donnent de la variété et rendent le match équitable.

On en déduit un **Elo** par maximum de vraisemblance, **ancré sur les niveaux Stockfish** (1320/1600/1900),
avec un intervalle de confiance à 95 % par bootstrap. Toutes les parties sont exportées en PGN.

⚠️ L'Elo « UCI_Elo » de Stockfish est calibré à cadence longue contre des moteurs (échelle CCRL) : c'est un ordre de grandeur,
pas un Elo Lichess ou FIDE. Les **écarts entre tes modèles** sont eux fiables.

⚙️ Réglages Kaggle : **Internet ON**, **GPU T4** conseillé. Durée : ~2–4 h avec les réglages par défaut (reprise automatique si coupure).

In [ ]:
# onnxruntime-gpu remplace onnxruntime (les deux ne cohabitent pas) ; marche aussi sans GPU.
# Version fixée : les plus récentes demandent CUDA 13, Kaggle a CUDA 12. [cuda,cudnn] installe les bibliothèques CUDA 12.
!pip uninstall -y -q onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q chess zstandard "onnxruntime-gpu[cuda,cudnn]==1.24.1"

In [ ]:
%%writefile commun_echecs.py
"""Code commun aux notebooks de benchmark échecs (PAWN, PAWN big, contrôle, SPAWN/PWN).

Tout vient de pawn_app.py : même encodage 69 octets, même recherche « arbre complet »,
même alpha-bêta + quiescence. Seule différence : pas de bruit ni de coups au hasard,
on veut mesurer la force réelle.
"""
import glob
import os
import shutil
import stat
import subprocess
import tarfile
import time
import urllib.request

import chess
import chess.engine
import chess.polyglot
import numpy as np
import onnxruntime as ort

DEPOT = "LiamLitle/ia-de-liam"
BRANCHE = "main"
LFS = f"https://media.githubusercontent.com/media/{DEPOT}/{BRANCHE}/"
DOSSIER_POIDS = os.environ.get("PAWN_POIDS", "/kaggle/working/poids" if os.path.isdir("/kaggle") else "poids")
MATE = 100_000

# nom -> chemin dans le dépôt
RESEAUX = {
    "PAWN": "Pawn/pawn.onnx",
    "PAWN_BIG": "PawnBig-V1/pawn_big.onnx",
    "CONTROLE": "PawnBig-controlleur/pawn_big_controle.onnx",
    "SPAWN": "PWN-soluce/pawn_soluce.onnx",
}


def poids(chemin_depot):
    """cherche le fichier dans /kaggle/input (dataset ajouté au notebook), sinon le télécharge depuis GitHub LFS"""
    nom = os.path.basename(chemin_depot)
    for racine in ("/kaggle/input", DOSSIER_POIDS):
        trouves = glob.glob(os.path.join(racine, "**", nom), recursive=True)
        trouves = [t for t in trouves if os.path.getsize(t) > 1000]  # pas un pointeur LFS
        if trouves:
            return trouves[0]
    os.makedirs(DOSSIER_POIDS, exist_ok=True)
    dest = os.path.join(DOSSIER_POIDS, nom)
    print(f"téléchargement de {chemin_depot} ...")
    urllib.request.urlretrieve(LFS + chemin_depot, dest)
    if os.path.getsize(dest) < 1000:
        raise RuntimeError(f"{dest} ressemble à un pointeur LFS, pas au vrai fichier")
    return dest


GPU_ID = 0       # carte utilisée par ce processus (Kaggle « GPU T4 x2 » en a deux)
GPU_MEM_MO = 0   # plafond mémoire GPU par réseau (0 = pas de plafond) ; fixé quand plusieurs processus partagent le GPU
_CUDA_OK = None  # None = pas encore essayé, False = échec -> on reste sur CPU sans réessayer


def providers(gpu=True):
    dispo = ort.get_available_providers()
    if gpu and _CUDA_OK is not False and "CUDAExecutionProvider" in dispo:
        try:
            ort.preload_dlls()  # charge CUDA/cuDNN depuis les paquets pip nvidia-* (déjà là avec torch sur Kaggle)
        except Exception:
            pass
        # HEURISTIC : cuDNN choisit ses algos sans essayer ceux qui demandent d'énormes
        # zones de travail (sinon des blocs de 150 Mo dépassent le plafond mémoire)
        opts = {"arena_extend_strategy": "kSameAsRequested", "cudnn_conv_algo_search": "HEURISTIC",
                "cudnn_conv_use_max_workspace": "0", "device_id": GPU_ID}
        if GPU_MEM_MO:
            opts["gpu_mem_limit"] = GPU_MEM_MO * 1024 * 1024
        return [("CUDAExecutionProvider", opts), "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


PIECES_ID = {c: i + 1 for i, c in enumerate("PNBRQK")}


def enc_fen(fen):
    f = fen.split(" ")
    noir = f[1] == "b"
    out = bytearray(69)
    r = c = 0
    for ch in f[0]:
        if ch == "/":
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            nous = ch.isupper() != noir
            out[(r if noir else 7 - r) * 8 + c] = PIECES_ID[ch.upper()] + (0 if nous else 6)
            c += 1
    ours, theirs = ("kq", "KQ") if noir else ("KQ", "kq")
    out[64] = ours[0] in f[2]
    out[65] = ours[1] in f[2]
    out[66] = theirs[0] in f[2]
    out[67] = theirs[1] in f[2]
    if f[3] != "-":
        out[68] = ord(f[3][0]) - 96
    return bytes(out)


class Evaluateur:
    """un réseau ONNX : liste de FEN -> centipions du point de vue du camp au trait"""

    def __init__(self, nom, gpu=True, threads=0):
        self.nom = nom
        self.so = ort.SessionOptions()
        if threads:
            self.so.intra_op_num_threads = threads
            self.so.inter_op_num_threads = 1
        global _CUDA_OK
        prov = providers(gpu)
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), self.so, providers=prov)
        self.sur_gpu = "CUDAExecutionProvider" in self.session.get_providers()
        if prov[0] != "CPUExecutionProvider":
            _CUDA_OK = self.sur_gpu
            if not _CUDA_OK:
                print("⚠️ GPU indisponible pour onnxruntime : on continue sur CPU (plus lent mais résultats identiques)")
                ort.set_default_logger_severity(4)
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        if self.sur_gpu and GPU_MEM_MO:
            lot = min(lot, 512)  # petits lots quand la mémoire GPU est plafonnée
        res = []
        for i in range(0, len(fens), lot):
            x = np.frombuffer(b"".join(map(enc_fen, fens[i:i + lot])), np.uint8).reshape(-1, 69).copy()
            try:
                res.append(self.session.run(None, {"board": x})[0])
            except Exception as e:
                if not self.sur_gpu:
                    raise
                # GPU plein : ce réseau passe sur CPU pour la suite (mêmes résultats, plus lent)
                print(f"⚠️ {self.nom} : erreur GPU ({str(e)[:80]}...) -> CPU")
                self.session = ort.InferenceSession(poids(RESEAUX[self.nom]), self.so, providers=["CPUExecutionProvider"])
                self.sur_gpu = False
                res.append(self.session.run(None, {"board": x})[0])
        self.nb_evals += len(fens)
        return np.concatenate(res)

    def eval1(self, fen):
        # l'alpha-bêta réévalue souvent les mêmes positions en quiescence : le réseau est
        # déterministe, donc un cache ne change rien au résultat, juste la vitesse
        v = self.cache.get(fen)
        if v is None:
            if len(self.cache) > 2_000_000:
                self.cache.clear()
            v = self.cache[fen] = float(self.evals([fen])[0])
        return v


# -- recherche « arbre complet » (PAWN, PAWN big, contrôle, SPAWN) -----------------

def arbre(board, depth, ply, feuilles):
    if not any(board.legal_moves):
        return ("t", -(MATE - ply) if board.is_check() else 0)
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return ("t", 0)
    if depth == 0:
        feuilles.append(board.fen())
        return ("f", len(feuilles) - 1)
    fils = []
    for mv in list(board.legal_moves):
        board.push(mv)
        fils.append((mv, arbre(board, depth - 1, ply + 1, feuilles)))
        board.pop()
    return ("n", fils)


def valeur(noeud, v):
    genre, x = noeud
    if genre == "t":
        return x
    if genre == "f":
        return v[x]
    return max(-valeur(c, v) for _, c in x)


def choisir_arbre(board, ev, depth):
    feuilles = []
    racine = arbre(board, depth, 0, feuilles)
    v = ev.evals(feuilles) if feuilles else []
    notes = [(-valeur(c, v), mv) for mv, c in racine[1]]
    return max(notes, key=lambda t: t[0])[1]


# -- recherche alpha-bêta + quiescence (PWN) --------------------------------------

VAL_PIECE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}


def trier_coups(board, coups, priorite):
    def cle(mv):
        if mv == priorite:
            return 10_000
        if board.is_capture(mv):
            prise = board.piece_type_at(mv.to_square) or chess.PAWN
            attaquant = board.piece_type_at(mv.from_square)
            return 1000 + VAL_PIECE[prise] * 10 - VAL_PIECE[attaquant]
        if board.gives_check(mv):
            return 500
        return 0
    return sorted(coups, key=cle, reverse=True)


def quiescence(board, ev, alpha, beta, prof_q):
    stand_pat = ev.eval1(board.fen())
    if stand_pat >= beta:
        return beta
    alpha = max(alpha, stand_pat)
    if prof_q <= 0:
        return alpha
    captures = [m for m in board.legal_moves if board.is_capture(m)]
    for mv in trier_coups(board, captures, None):
        board.push(mv)
        score = -quiescence(board, ev, -beta, -alpha, prof_q - 1)
        board.pop()
        if score >= beta:
            return beta
        alpha = max(alpha, score)
    return alpha


def negamax(board, ev, profondeur, alpha, beta, ply, prof_q, tt, coup_tt):
    alpha0 = alpha
    if not any(board.legal_moves):
        return -(MATE - ply) if board.is_check() else 0
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return 0
    hash_ = chess.polyglot.zobrist_hash(board)
    entree = tt.get(hash_)
    if entree and entree[0] >= profondeur:
        d, v, drapeau, _ = entree
        if drapeau == "exact":
            return v
        if drapeau == "min":
            alpha = max(alpha, v)
        elif drapeau == "max":
            beta = min(beta, v)
        if alpha >= beta:
            return v
    if profondeur <= 0:
        return quiescence(board, ev, alpha, beta, prof_q)
    meilleur, meilleur_coup = -MATE - 1, None
    for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
        board.push(mv)
        score = -negamax(board, ev, profondeur - 1, -beta, -alpha, ply + 1, prof_q, tt, coup_tt)
        board.pop()
        if score > meilleur:
            meilleur, meilleur_coup = score, mv
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
    tt[hash_] = (profondeur, meilleur, drapeau, meilleur_coup)
    if meilleur_coup is not None:
        coup_tt[hash_] = meilleur_coup
    return meilleur


def choisir_alphabeta(board, ev, depth, prof_q=4):
    tt, coup_tt = {}, {}
    meilleur_coup = None
    for d in range(1, depth + 1):
        alpha, beta = -MATE - 1, MATE + 1
        hash_ = chess.polyglot.zobrist_hash(board)
        meilleur_score, meilleur_du_tour = -MATE - 1, None
        for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
            board.push(mv)
            score = -negamax(board, ev, d - 1, -beta, -alpha, 1, prof_q, tt, coup_tt)
            board.pop()
            if score > meilleur_score:
                meilleur_score, meilleur_du_tour = score, mv
            alpha = max(alpha, score)
        meilleur_coup = meilleur_du_tour
        coup_tt[hash_] = meilleur_coup
    return meilleur_coup


# -- joueurs ------------------------------------------------------------------------
# un « joueur » = un réseau + un algorithme de recherche + une profondeur.
# ex : "PAWN_BIG@arbre2", "SPAWN@arbre1", "PWN@ab3" (PWN = poids SPAWN + alpha-bêta).

def decoder(joueur):
    nom, rech = joueur.split("@")
    if nom == "PWN":
        nom = "SPAWN"
    if rech.startswith("arbre"):
        return nom, "arbre", int(rech[5:])
    if rech.startswith("ab"):
        return nom, "ab", int(rech[2:])
    raise ValueError(joueur)


_EVALS = {}
_SF = {}
GPU = True
THREADS = 0
SF_CHEMIN = None
SF_TEMPS = 0.1


def evaluateur(nom, gpu=None):
    gpu = GPU if gpu is None else gpu
    if (nom, gpu) not in _EVALS:
        _EVALS[(nom, gpu)] = Evaluateur(nom, gpu=gpu, threads=THREADS)
    return _EVALS[(nom, gpu)]


def configurer(gpu=True, threads=0, sf_chemin=None, sf_temps=0.1):
    """à appeler dans chaque processus fils avant de jouer"""
    global GPU, THREADS, SF_CHEMIN, SF_TEMPS
    GPU, THREADS, SF_CHEMIN, SF_TEMPS = gpu, threads, sf_chemin, sf_temps
    _EVALS.clear()
    _SF.clear()


def stockfish(cfg):
    """cfg = "elo1500" (UCI_LimitStrength) ou "skill0" (Skill Level)"""
    if cfg not in _SF:
        e = chess.engine.SimpleEngine.popen_uci(SF_CHEMIN)
        opts = {"Threads": 1, "Hash": 16}
        if cfg.startswith("elo"):
            opts.update({"UCI_LimitStrength": True, "UCI_Elo": int(cfg[3:])})
        elif cfg.startswith("skill"):
            opts["Skill Level"] = int(cfg[5:])
        e.configure(opts)
        _SF[cfg] = e
    return _SF[cfg]


def fermer_stockfish():
    for e in _SF.values():
        e.quit()
    _SF.clear()


def coup(joueur, board):
    if joueur.startswith("SF@"):
        return stockfish(joueur[3:]).play(board, chess.engine.Limit(time=SF_TEMPS)).move
    nom, rech, depth = decoder(joueur)
    b = board.copy(stack=True)
    if rech == "arbre":
        return choisir_arbre(b, evaluateur(nom), depth)
    # l'alpha-bêta évalue une position à la fois : sur Kaggle la latence CPU (~6 ms)
    # est deux fois plus basse que celle du GPU (~14 ms), donc on reste sur CPU
    return choisir_alphabeta(b, evaluateur(nom, gpu=False), depth)


# -- Stockfish -------------------------------------------------------------------------

URLS_STOCKFISH = [
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17.1/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_16.1/stockfish-ubuntu-x86-64-avx2.tar",
]


def rendre_executable(binaire):
    """/kaggle/working interdit d'exécuter des programmes : on copie Stockfish ailleurs
    et on garde le premier emplacement où il démarre vraiment"""
    for dest in ("/usr/local/bin/stockfish-pawn", "/tmp/stockfish-pawn", binaire):
        try:
            if dest != binaire:
                shutil.copy(binaire, dest)
            os.chmod(dest, 0o755)
            subprocess.run([dest, "quit"], check=True, timeout=10, capture_output=True)
            return dest
        except Exception as e:
            print("Stockfish ne démarre pas depuis", dest, ":", e)
    raise RuntimeError("Stockfish ne peut être exécuté nulle part")


def installer_stockfish(dossier=None):
    dossier = dossier or os.path.join(os.path.dirname(os.path.abspath(DOSSIER_POIDS)), "stockfish")
    deja = glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
    deja = [d for d in deja if os.path.isfile(d) and not d.endswith(".tar")]
    if deja:
        return rendre_executable(deja[0])
    os.makedirs(dossier, exist_ok=True)
    for url in URLS_STOCKFISH:
        try:
            tar = os.path.join(dossier, "sf.tar")
            urllib.request.urlretrieve(url, tar)
            with tarfile.open(tar) as t:
                t.extractall(dossier)
            binaire = [d for d in glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
                       if os.path.isfile(d) and not d.endswith(".tar")][0]
            return rendre_executable(binaire)
        except Exception as e:
            print("échec", url, e)
    if shutil.which("stockfish") is None:
        subprocess.run("apt-get -qq install -y stockfish", shell=True)
    for p in (shutil.which("stockfish"), "/usr/games/stockfish"):
        if p and os.path.exists(p):
            return p
    raise RuntimeError("impossible d'installer Stockfish")


# -- puzzles Lichess ---------------------------------------------------------------

def resoudre_puzzle(joueur, fen, moves):
    """format Lichess : FEN avant le coup adverse, moves[0] = coup adverse, puis solution.
    Comme sur Lichess, un mat est toujours accepté même si ce n'est pas le coup attendu.
    renvoie (résolu, premier_coup_bon, nb_coups_joués, secondes)"""
    b = chess.Board(fen)
    moves = moves.split()
    b.push_uci(moves[0])
    premier, joues, t0 = None, 0, time.perf_counter()
    for i in range(1, len(moves), 2):
        attendu = chess.Move.from_uci(moves[i])
        mv = coup(joueur, b)
        joues += 1
        b.push(mv)
        mat = b.is_checkmate()
        b.pop()
        ok = mv == attendu or mat
        if premier is None:
            premier = ok
        if not ok:
            return False, premier, joues, time.perf_counter() - t0
        if mat:
            break
        b.push(attendu)
        if i + 1 < len(moves):
            b.push_uci(moves[i + 1])
    return True, premier, joues, time.perf_counter() - t0


# -- parties -------------------------------------------------------------------------

def jouer_partie(blancs, noirs, ouverture, max_plies=300):
    """ouverture = liste de coups UCI joués d'office ; renvoie un dict (résultat, pgn, temps)"""
    import chess.pgn
    b = chess.Board()
    for u in ouverture:
        b.push_uci(u)
    temps = {blancs: 0.0, noirs: 0.0}
    nb = {blancs: 0, noirs: 0}
    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
        j = blancs if b.turn == chess.WHITE else noirs
        t0 = time.perf_counter()
        mv = coup(j, b)
        temps[j] += time.perf_counter() - t0
        nb[j] += 1
        b.push(mv)
    res = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
    fin = b.outcome(claim_draw=True)
    jeu = chess.pgn.Game.from_board(b)
    jeu.headers.update(Event="Benchmark P.A.W.N.", White=blancs, Black=noirs, Result=res)
    return {
        "blancs": blancs, "noirs": noirs, "resultat": res,
        "fin": fin.termination.name if fin else "MAX_PLIES",
        "plies": b.ply(), "pgn": str(jeu),
        "s_par_coup_blancs": temps[blancs] / max(1, nb[blancs]),
        "s_par_coup_noirs": temps[noirs] / max(1, nb[noirs]),
    }


# -- tâches pour ProcessPoolExecutor (doivent être importables depuis ce module) -------

def nb_gpu():
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        return max(1, sum(l.startswith("GPU") for l in r.stdout.splitlines()))
    except Exception:
        return 1


def init_worker(gpu, threads, sf_chemin=None, sf_temps=0.1, gpu_mem_mo=None):
    # plusieurs processus x plusieurs réseaux sur un GPU : sans plafond, onnxruntime réserve
    # la mémoire trop largement et le T4 sature (BFCArena / CUBLAS_STATUS_ALLOC_FAILED).
    # Les processus sont répartis sur les GPU disponibles, avec un plafond par réseau.
    global GPU_MEM_MO, GPU_ID
    n = nb_gpu()
    GPU_ID = os.getpid() % n
    GPU_MEM_MO = gpu_mem_mo or (700 if n == 1 else 1200)
    configurer(gpu=gpu, threads=threads, sf_chemin=sf_chemin, sf_temps=sf_temps)


def tache_puzzle(args):
    joueur, pid, fen, moves = args
    ok, premier, joues, sec = resoudre_puzzle(joueur, fen, moves)
    return {"joueur": joueur, "PuzzleId": pid, "resolu": ok, "premier_coup": premier,
            "coups_joues": joues, "secondes": sec}


def tache_partie(args):
    blancs, noirs, ouverture, max_plies, id_ouv = args
    try:
        r = jouer_partie(blancs, noirs, ouverture, max_plies)
    finally:
        # un Stockfish ouvert garde un thread vivant qui empêche le processus fils de se terminer
        fermer_stockfish()
    r["ouverture"] = id_ouv
    return r

In [ ]:
import os, sys, time, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import chess
sys.path.insert(0, os.getcwd())  # pour que les processus fils trouvent commun_echecs.py
import onnxruntime as ort
import commun_echecs as ce

print("onnxruntime", ort.__version__, "| providers dispo :", ort.get_available_providers())
ev_test = ce.Evaluateur("PAWN")
print("session PAWN sur :", ev_test.session.get_providers())
print("éval position initiale :", ev_test.eval1(chess.STARTING_FEN), "cp")

In [ ]:
SF = ce.installer_stockfish(os.path.join(os.getcwd(), "stockfish"))
print("Stockfish :", SF)

In [ ]:
# ---- réglages ----
MODELES = ["PAWN@arbre2", "PAWN_BIG@arbre2", "CONTROLE@arbre2", "SPAWN@arbre2", "PWN@ab2"]
# "PWN@ab3" est plus fort mais ~5x plus lent : à ajouter si tu as le temps
STOCKFISH = {"SF@elo1320": 1320, "SF@elo1600": 1600, "SF@elo1900": 1900}   # ancres Elo
STOCKFISH_LIBRES = ["SF@skill0"]                                             # joués mais pas ancrés
SF_TEMPS = 0.1          # secondes par coup pour Stockfish
N_OUVERTURES = 8        # ouvertures par paire (x2 couleurs)
MAX_PLIES = 300         # au-delà : nulle
GPU = True
NB_WORKERS = os.cpu_count()
THREADS = 1
BUDGET_H = 9            # on arrête de lancer des parties après ce temps (limite Kaggle : 12 h)
SEED = 0
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

OUVERTURES = {
    "Italienne": "e2e4 e7e5 g1f3 b8c6 f1c4 f8c5",
    "Espagnole": "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6",
    "Sicilienne Najdorf": "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 f3d4 g8f6 b1c3 a7a6",
    "Française": "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6",
    "Caro-Kann": "e2e4 c7c6 d2d4 d7d5 e4e5 c8f5",
    "Gambit dame refusé": "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6",
    "Slave": "d2d4 d7d5 c2c4 c7c6 g1f3 g8f6",
    "Est-indienne": "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 e2e4 d7d6",
    "Nimzo-indienne": "d2d4 g8f6 c2c4 e7e6 b1c3 f8b4",
    "Anglaise": "c2c4 e7e5 b1c3 g8f6 g1f3 b8c6",
    "Scandinave": "e2e4 d7d5 e4d5 d8d5 b1c3 d5a5",
    "Londres": "d2d4 d7d5 g1f3 g8f6 c1f4 c7c5",
    "Pirc": "e2e4 d7d6 d2d4 g8f6 b1c3 g7g6",
    "Hollandaise": "d2d4 f7f5 g1f3 g8f6 g2g3 e7e6",
    "Écossaise": "e2e4 e7e5 g1f3 b8c6 d2d4 e5d4 f3d4 g8f6",
    "Réti": "g1f3 d7d5 g2g3 g8f6 f1g2 e7e6",
}
for nom, coups in OUVERTURES.items():  # vérifie que les ouvertures sont légales
    b = chess.Board()
    for u in coups.split():
        assert chess.Move.from_uci(u) in b.legal_moves, (nom, u)
        b.push_uci(u)
OUV = dict(list(OUVERTURES.items())[:N_OUVERTURES])

In [ ]:
from itertools import combinations
paires = list(combinations(MODELES, 2)) + [(m, s) for m in MODELES for s in list(STOCKFISH) + STOCKFISH_LIBRES]
taches = []
for a, b in paires:
    for nom, coups in OUV.items():
        taches.append((a, b, coups.split(), MAX_PLIES, nom))
        taches.append((b, a, coups.split(), MAX_PLIES, nom))
random.Random(SEED).shuffle(taches)  # si ça coupe, les résultats partiels restent équilibrés
print(len(paires), "paires,", len(taches), "parties")

## Parties (en parallèle, avec reprise)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

PARTIES = f"{SORTIE}/bench3_parties.jsonl"
fait = set()
if os.path.exists(PARTIES):
    for l in open(PARTIES):
        r = json.loads(l); fait.add((r["blancs"], r["noirs"], r["ouverture"]))
reste = [t for t in taches if (t[0], t[1], t[4]) not in fait]
print(len(fait), "déjà jouées,", len(reste), "à jouer")

t0 = time.time()
ex = ProcessPoolExecutor(NB_WORKERS, mp_context=mp.get_context("spawn"),
                         initializer=ce.init_worker, initargs=(GPU, THREADS, SF, SF_TEMPS))
futurs = [ex.submit(ce.tache_partie, t) for t in reste]
with open(PARTIES, "a") as f:
    for fu in tqdm(as_completed(futurs), total=len(futurs)):
        try:
            f.write(json.dumps(fu.result()) + "\n"); f.flush()
        except Exception as e:
            print("erreur :", e)
        if time.time() - t0 > BUDGET_H * 3600:
            print("budget temps atteint, on s'arrête là")
            for x in futurs:
                x.cancel()
            break
ex.shutdown(wait=False, cancel_futures=True)

## Résultats

In [ ]:
parties = pd.DataFrame([json.loads(l) for l in open(PARTIES)])
parties = parties.drop_duplicates(["blancs", "noirs", "ouverture"])
parties["score_blancs"] = parties.resultat.map({"1-0": 1.0, "0-1": 0.0, "1/2-1/2": 0.5})
with open(f"{SORTIE}/bench3_parties.pgn", "w") as f:
    f.write("\n\n".join(parties.pgn))
print(len(parties), "parties |", parties.fin.value_counts().to_dict())

joueurs = MODELES + list(STOCKFISH) + STOCKFISH_LIBRES
croise = pd.DataFrame(np.nan, index=joueurs, columns=joueurs)
for a in joueurs:
    for b in joueurs:
        g1 = parties[(parties.blancs == a) & (parties.noirs == b)].score_blancs
        g2 = 1 - parties[(parties.blancs == b) & (parties.noirs == a)].score_blancs
        s = pd.concat([g1, g2])
        if len(s):
            croise.loc[a, b] = s.sum()
print("score de la ligne contre la colonne (sur", 2 * len(OUV), "parties)")
croise.loc[MODELES].dropna(axis=1, how="all")

In [ ]:
from scipy.optimize import minimize

def elo_mle(df, ancres, joueurs, prior_sd=1000):
    """Bradley-Terry / Elo par maximum de vraisemblance (nulle = ½ victoire) ; joueurs ancrés à leur Elo"""
    libres = [j for j in joueurs if j not in ancres]
    idx = {j: i for i, j in enumerate(libres)}
    w, n = df.blancs.to_numpy(), df.noirs.to_numpy()
    s = df.score_blancs.to_numpy()
    moy = np.mean(list(ancres.values()))
    def notes(x):
        return np.array([ancres[j] if j in ancres else x[idx[j]] for j in w]), \
               np.array([ancres[j] if j in ancres else x[idx[j]] for j in n])
    def nll(x):
        rw, rn = notes(x)
        p = np.clip(1 / (1 + 10 ** ((rn - rw) / 400)), 1e-9, 1 - 1e-9)
        return -np.sum(s * np.log(p) + (1 - s) * np.log(1 - p)) + np.sum((x - moy) ** 2) / (2 * prior_sd ** 2)
    x = minimize(nll, np.full(len(libres), moy), method="L-BFGS-B").x
    return {**{j: x[idx[j]] for j in libres}, **ancres}

elo = elo_mle(parties, STOCKFISH, joueurs)
rng = np.random.default_rng(SEED)
boot = []
for _ in range(200):
    boot.append(elo_mle(parties.sample(len(parties), replace=True, random_state=int(rng.integers(1e9))), STOCKFISH, joueurs))
boot = pd.DataFrame(boot)

classement = pd.DataFrame({
    "Elo": pd.Series(elo),
    "IC 95 % bas": boot.quantile(0.025),
    "IC 95 % haut": boot.quantile(0.975),
})
score = {}
for j in joueurs:
    s = pd.concat([parties[parties.blancs == j].score_blancs, 1 - parties[parties.noirs == j].score_blancs])
    score[j] = (s.mean() * 100, len(s))
classement["score (%)"] = [score[j][0] for j in classement.index]
classement["parties"] = [score[j][1] for j in classement.index]
vit = pd.concat([parties.groupby("blancs").s_par_coup_blancs.mean(), parties.groupby("noirs").s_par_coup_noirs.mean()], axis=1).mean(axis=1)
classement["s / coup"] = vit
classement = classement.sort_values("Elo", ascending=False)
classement.to_csv(f"{SORTIE}/bench3_classement.csv")
classement.round(1)

In [ ]:
c = classement.loc[[j for j in classement.index if j in MODELES or j in STOCKFISH_LIBRES]]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(c.index, c.Elo, xerr=[c.Elo - c["IC 95 % bas"], c["IC 95 % haut"] - c.Elo], color="#58a", capsize=4)
for niveau, e in STOCKFISH.items():
    ax.axvline(e, ls="--", color="gray", lw=1); ax.text(e, len(c) - 0.4, niveau.replace("SF@", "SF "), fontsize=8, ha="center")
ax.invert_yaxis(); ax.set_xlabel("Elo (ancré sur Stockfish UCI_Elo)"); ax.set_title("Classement Elo des modèles")
plt.tight_layout(); plt.savefig(f"{SORTIE}/bench3_elo.png", dpi=120); plt.show()

### Comment lire les résultats
- Avec 16 parties par paire l'IC est large (±80–150 Elo) : pour trancher entre deux modèles proches, augmente `N_OUVERTURES`.
- Si un modèle fait 0 % ou 100 % contre un niveau Stockfish, son Elo repose sur les autres niveaux : ajoute un niveau plus fort/faible (`SF@elo2200`, `SF@elo1450`…).
- `fin` = `MAX_PLIES` souvent → les moteurs tournent en rond en finale (signe classique d'une éval sans notion de progrès).